# Dunnhumby M5-C seed44 동일시드 재현

seed42에서 M1 대비 정확도·경제성과를 같이 높였지만 M3의 Top-10·@20 이득을 보존하지 못한 M5-C를 **수식·강도·epoch를 바꾸지 않고** seed44에서 한 번 재현합니다.

- 새 학습: M5-C **1회**
- 재사용: 동일 seed44 M1·M3·M5-A·M5-B
- 고정: Dunnhumby 개발 684~690, seed44, 300 epoch, 판정은 300 epoch
- M4-C: `exp(0.5×q_C×(상품경제항-고객별평균))`, 고객별 평균 가중치 1
- final test·holdout 미사용, M4-C 단독 미학습, epoch·강도 선택 없음

판정: 두 경제지표@10이 M1·M3보다 모두 높고, 여섯 정확도가 M1·M3 각각의 99% 이상인지를 본다. M5-A·B와의 비교는 동일 seed44의 기술적 비교이며 유의성·일반화를 주장하지 않는다.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, os, subprocess, sys

if not os.path.ismount('/content/drive'):
    try:
        drive.mount('/content/drive')
    except (ValueError, NotImplementedError) as exc:
        raise RuntimeError('Drive 연결 실패. 학습을 시작하지 않았습니다.') from exc

ROOT = Path('/content/drive/MyDrive/논문/data')
OLD = ROOT/'results_v3_dunnhumby_clv_m5_m3_m4_split_nv_s44_v1'
if len(list(OLD.glob('clv-m5-m3-m4-split-nv-dev-v1_*.json'))) != 1:
    raise RuntimeError(f'동일 seed44 M5-A/B 완료 결과가 필요합니다: {OLD} — 새 학습 전에 중단했습니다.')

SOURCE_COMMIT = 'ec2b1ae949a9b04e3bef0919b3453dfb0edba637'
REPO = Path('/content/clv-m5-m4c-s44-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', SOURCE_COMMIT], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
os.chdir(REPO)
sys.path.insert(0, str(REPO))

import torch
assert torch.cuda.is_available(), '학습에는 GPU 런타임을 사용하세요.'
import clv_m5_m3_m4_user_centered_screen as screen
screen.self_test()
cfg = screen.configure(seed=44)
print(json.dumps({'source_commit': SOURCE_COMMIT, 'new_fit': screen.MODEL_ID, 'reused': ['M1','M3','M5-A','M5-B'], 'seed': 44, 'epochs': 300, 'out_dir': cfg.out_dir}, ensure_ascii=False, indent=2))

## M5-C 1회 학습

런타임이 중단되면 같은 셀을 다시 실행하세요. 완료 결과는 재사용하고 미완료 학습은 checkpoint에서 재개합니다.

In [ ]:
result = screen.run(cfg)

In [ ]:
from IPython.display import display
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

metrics = ['recall@10','ndcg@10','recall@20','ndcg@20','recall@50','ndcg@50',
           'price_purchase_amount_weighted_hit@10','vndcg@10',
           'price_purchase_amount_weighted_hit@20','vndcg@20',
           'price_purchase_amount_weighted_hit@50','vndcg@50']
display(result['absolute'].query('epoch == 300')[['model_id','seed','epoch',*metrics]])
display(result['comparison'].query('epoch == 300 and metric in @metrics'))
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))

zip_path = Path('/content/clv_m5_m3_m4_user_centered_seed44_results.zip')
with ZipFile(zip_path, 'w', ZIP_DEFLATED) as archive:
    for path in result['paths'].values():
        archive.write(path, Path(path).name)
files.download(str(zip_path))